# Boostrap cell (run to detect environment local/colab)

In [ ]:
# ============================================================
# PROJECT BOOTSTRAP
# Local  : dùng project + data trên máy
# Colab : dùng project/data trên Google Drive
#         + /content/cache làm vùng xử lý nhanh tạm thời
# ============================================================

from pathlib import Path
import sys
import shutil


# ============================================================
# 0. CONFIG - CHỈ CẦN SỬA PHẦN NÀY
# ============================================================

# Project trên Google Drive khi chạy bằng Colab
COLAB_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/projects/santander"
)

# Tên folder dùng để xác định project root khi chạy local
PROJECT_MARKERS = [
    "src",
    "notebooks",
]


# ============================================================
# 1. DETECT ENVIRONMENT
# ============================================================

IS_COLAB = "google.colab" in sys.modules
ENV = "colab" if IS_COLAB else "local"


# ============================================================
# 2. HELPER: FIND LOCAL PROJECT ROOT
# ============================================================

def find_project_root(start: Path) -> Path:
    """
    Đi ngược từ current working directory cho đến khi tìm thấy
    thư mục project chứa các marker như src/, notebooks/.
    """
    current = start.resolve()

    while True:
        if any((current / marker).exists() for marker in PROJECT_MARKERS):
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        f"Không tìm thấy project root từ: {start}"
    )


# ============================================================
# 3. ENVIRONMENT-SPECIFIC SETUP
# ============================================================

if IS_COLAB:

    # --------------------------------------------------------
    # Mount Google Drive
    # --------------------------------------------------------
    from google.colab import drive

    DRIVE_MOUNT = Path("/content/drive")

    if not (DRIVE_MOUNT / "MyDrive").exists():
        drive.mount(str(DRIVE_MOUNT))

    PROJECT_ROOT = COLAB_PROJECT_ROOT

    # Persistent data nằm trên Drive
    DATA_ROOT = PROJECT_ROOT / "data"

    # Temporary fast storage của Colab
    WORK_ROOT = Path("/content/cache")

else:

    # --------------------------------------------------------
    # Local
    # --------------------------------------------------------
    PROJECT_ROOT = find_project_root(Path.cwd())

    DATA_ROOT = PROJECT_ROOT / "data"

    # Local có thể dùng interim như working area luôn
    WORK_ROOT = PROJECT_ROOT / ".cache"


# ============================================================
# 4. COMMON PATHS
# ============================================================

SRC_DIR = PROJECT_ROOT / "src"

RAW_DIR = DATA_ROOT / "raw"
INTERIM_DIR = DATA_ROOT / "interim"
PROCESSED_DIR = DATA_ROOT / "processed"

WORK_RAW_DIR = WORK_ROOT / "raw"
WORK_INTERIM_DIR = WORK_ROOT / "interim"
WORK_PROCESSED_DIR = WORK_ROOT / "processed"

DUCKDB_TEMP_DIR = WORK_ROOT / "duckdb_temp"


# ============================================================
# 5. CREATE DIRECTORIES
# ============================================================

directories = [
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    WORK_RAW_DIR,
    WORK_INTERIM_DIR,
    WORK_PROCESSED_DIR,
    DUCKDB_TEMP_DIR,
]

for directory in directories:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 6. MAKE src/ IMPORTABLE
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


# ============================================================
# 7. FILE STAGING HELPERS
# ============================================================

def stage_file(
    source: Path,
    destination_dir: Path = WORK_INTERIM_DIR,
    overwrite: bool = False,
) -> Path:
    """
    Copy file từ persistent storage sang working storage.

    Colab:
        Google Drive -> /content/cache

    Local:
        project data -> .cache

    Trả về path của file trong working storage.
    """

    source = Path(source)

    if not source.exists():
        raise FileNotFoundError(
            f"Source file không tồn tại: {source}"
        )

    destination_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    destination = destination_dir / source.name

    if destination.exists() and not overwrite:
        return destination

    shutil.copy2(
        source,
        destination,
    )

    return destination


def persist_file(
    source: Path,
    destination_dir: Path = PROCESSED_DIR,
    overwrite: bool = True,
) -> Path:
    """
    Copy output từ working storage về persistent storage.

    Colab:
        /content/cache -> Google Drive

    Local:
        .cache -> data/processed
    """

    source = Path(source)

    if not source.exists():
        raise FileNotFoundError(
            f"Source file không tồn tại: {source}"
        )

    destination_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    destination = destination_dir / source.name

    if destination.exists():

        if not overwrite:
            return destination

        destination.unlink()

    shutil.copy2(
        source,
        destination,
    )

    return destination


# ============================================================
# 8. ENABLE AUTO-RELOAD IN NOTEBOOK
# ============================================================

try:
    ipython = get_ipython()

    ipython.run_line_magic(
        "load_ext",
        "autoreload",
    )

    ipython.run_line_magic(
        "autoreload",
        "2",
    )

except NameError:
    pass


# ============================================================
# 9. CONFIG SUMMARY
# ============================================================

print("=" * 70)
print("PROJECT ENVIRONMENT")
print("=" * 70)

print(f"Environment      : {ENV}")
print(f"Project root     : {PROJECT_ROOT}")
print(f"Source           : {SRC_DIR}")
print()

print("Persistent storage")
print(f"  Raw            : {RAW_DIR}")
print(f"  Interim        : {INTERIM_DIR}")
print(f"  Processed      : {PROCESSED_DIR}")
print()

print("Working storage")
print(f"  Work root      : {WORK_ROOT}")
print(f"  DuckDB temp    : {DUCKDB_TEMP_DIR}")

print("=" * 70)